In [ ]:
%pip install datasets transformers accelerate bitsandbytes huggingface_hub

# kubeflow SDK is installed by test harness via install_kubeflow.py

In [ ]:
import os
import urllib3
from kubernetes import client as k8s

# Suppress InsecureRequestWarning since we use verify_ssl = False
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN environment variables are required")

PVC_NAME = os.getenv("SHARED_PVC_NAME", "shared")

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = False
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)

PVC_MOUNT_PATH = "/opt/app-root/src"

In [ ]:
import json

from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer
from kubeflow.common.types import KubernetesBackendConfig

backend_cfg = KubernetesBackendConfig(
    client_configuration=api_client.configuration,
)

client = TrainerClient(backend_cfg)
print(client)

In [ ]:
# S3/MinIO and dataset download
import os
import gzip
import shutil
import time
import json
import random

try:
    import boto3
    from botocore.config import Config as BotoConfig
    from botocore.exceptions import ClientError
    HAS_BOTO3 = True
except ImportError:
    HAS_BOTO3 = False

PVC_NOTEBOOK_PATH = "/opt/app-root/src"
DATASET_ROOT_NOTEBOOK = PVC_NOTEBOOK_PATH
TABLE_GPT_DIR = os.path.join(DATASET_ROOT_NOTEBOOK, "table-gpt-data", "train")
MODEL_DIR = os.path.join(DATASET_ROOT_NOTEBOOK, "Qwen", "Qwen2.5-1.5B-Instruct")
os.makedirs(TABLE_GPT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

s3_endpoint = os.getenv("AWS_DEFAULT_ENDPOINT", "")
s3_access_key = os.getenv("AWS_ACCESS_KEY_ID", "")
s3_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY", "")
s3_bucket = os.getenv("AWS_STORAGE_BUCKET", "")
s3_prefix = os.getenv("AWS_STORAGE_BUCKET_SFT_DIR", "")

data_download_successful = False

def stream_download(s3, bucket, key, dst):
    """Download an object from S3/MinIO using streaming reads."""
    print(f"[notebook] STREAM download s3://{bucket}/{key} -> {dst}")
    t0 = time.time()
    try:
        resp = s3.get_object(Bucket=bucket, Key=key)
    except ClientError as e:
        print(f"[notebook] CLIENT ERROR for {key}: {e.response.get('Error', {})}")
        return False
    except Exception as e:
        print(f"[notebook] OTHER ERROR for {key}: {e}")
        return False

    body = resp["Body"]
    try:
        with open(dst, "wb") as f:
            while True:
                try:
                    chunk = body.read(1024 * 1024)
                except Exception as e:
                    print(f"[notebook] Error while reading {key}: {e}")
                    return False
                if not chunk:
                    break
                f.write(chunk)
    except Exception as e:
        print(f"[notebook] ERROR writing to {dst}: {e}")
        return False

    t1 = time.time()
    print(f"[notebook] DONE stream {key} in {t1 - t0:.2f}s")
    return True

if HAS_BOTO3 and s3_endpoint and s3_bucket:
    try:
        endpoint_url = s3_endpoint if s3_endpoint.startswith("http") else f"https://{s3_endpoint}"
        prefix = (s3_prefix or "").strip("/")

        print(f"[notebook] S3 configured: endpoint={endpoint_url}, bucket={s3_bucket}, prefix={prefix or '<root>'}")

        boto_cfg = BotoConfig(
            signature_version="s3v4",
            s3={"addressing_style": "path"},
            retries={"max_attempts": 1, "mode": "standard"},
            connect_timeout=5,
            read_timeout=10,
        )

        s3 = boto3.client(
            "s3",
            endpoint_url=endpoint_url,
            aws_access_key_id=s3_access_key,
            aws_secret_access_key=s3_secret_key,
            config=boto_cfg,
            verify=False,
        )

        paginator = s3.get_paginator("list_objects_v2")
        pulled_any = False
        file_count = 0

        print(f"[notebook] Starting S3 download from prefix: {prefix}")
        for page in paginator.paginate(Bucket=s3_bucket, Prefix=prefix or ""):
            contents = page.get("Contents", [])
            if not contents:
                print(f"[notebook] No contents found in this page")
                continue

            print(f"[notebook] Found {len(contents)} objects in this page")

            for obj in contents:
                key = obj["Key"]
                file_count += 1

                if key.endswith("/"):
                    print(f"[notebook] Skipping directory marker: {key}")
                    continue

                rel = key[len(prefix):].lstrip("/") if prefix else key
                print(f"[notebook] Processing key={key}, rel={rel}")

                if "table-gpt" in rel.lower() or rel.endswith(".jsonl"):
                    dst = os.path.join(TABLE_GPT_DIR, os.path.basename(rel))
                elif "qwen" in rel.lower() or any(rel.endswith(ext) for ext in [".bin", ".json", ".model", ".safetensors", ".txt"]):
                    dst = os.path.join(MODEL_DIR, rel.split("Qwen2.5-1.5B-Instruct/")[-1] if "Qwen2.5-1.5B-Instruct" in rel else os.path.basename(rel))
                else:
                    dst = os.path.join(DATASET_ROOT_NOTEBOOK, rel)

                os.makedirs(os.path.dirname(dst), exist_ok=True)

                if not os.path.exists(dst):
                    ok = stream_download(s3, s3_bucket, key, dst)
                    if not ok:
                        print(f"[notebook] Download failed for {key}")
                        continue
                    pulled_any = True
                else:
                    print(f"[notebook] Skipping existing file {dst}")
                    pulled_any = True

                if dst.endswith(".gz") and os.path.exists(dst):
                    out_path = os.path.splitext(dst)[0]
                    if not os.path.exists(out_path):
                        print(f"[notebook] Decompressing {dst} -> {out_path}")
                        try:
                            with gzip.open(dst, "rb") as f_in, open(out_path, "wb") as f_out:
                                shutil.copyfileobj(f_in, f_out)
                        except Exception as e:
                            print(f"[notebook] Failed to decompress {dst}: {e}")
                        else:
                            try:
                                os.remove(dst)
                            except Exception:
                                pass

        if pulled_any:
            print(f"[notebook] S3 download successful. Processed {file_count} files")
            data_download_successful = True
        else:
            print(f"[notebook] S3 download found no files to download")

    except Exception as e:
        print(f"[notebook] S3 fetch failed: {e}")
        import traceback
        traceback.print_exc()
        print("[notebook] Will attempt HuggingFace fallback...")
else:
    print("[notebook] S3 not configured (missing endpoint or bucket env vars)")

if not data_download_successful:
    print("[notebook] Attempting HuggingFace dataset download (requires internet)...")
    try:
        from datasets import load_dataset

        print("[notebook] Loading Table-GPT dataset from HuggingFace...")
        dataset = load_dataset("LipengCS/Table-GPT", "All")

        train_data = dataset["train"]
        print(f"[notebook] Original training set size: {len(train_data)}")

        random.seed(42)
        subset_indices = random.sample(range(len(train_data)), min(100, len(train_data)))
        subset_data = train_data.select(subset_indices)

        print(f"[notebook] Subset size: {len(subset_data)}")

        output_file = os.path.join(TABLE_GPT_DIR, "train_All_100.jsonl")
        with open(output_file, "w") as f:
            for example in subset_data:
                f.write(json.dumps(example) + "\n")

        print(f"[notebook] HuggingFace download successful. Subset saved to {output_file}")
        data_download_successful = True

    except Exception as hf_error:
        print(f"[notebook] HuggingFace download failed: {hf_error}")
        import traceback
        traceback.print_exc()
        raise RuntimeError(
            "Failed to download dataset from both S3 and HuggingFace. "
            "In disconnected environments, ensure S3/MinIO is configured with the required data. "
            "In connected environments, check your internet connection and credentials."
        ) from hf_error

dataset_file = os.path.join(TABLE_GPT_DIR, "train_All_100.jsonl")
if os.path.exists(dataset_file):
    print(f"[notebook] Dataset ready: {dataset_file}")
else:
    raise RuntimeError(f"Dataset file not found: {dataset_file}")

In [ ]:
# Model download - use S3 if available, otherwise HuggingFace
if os.path.exists(MODEL_DIR) and os.listdir(MODEL_DIR):
    model_path = MODEL_DIR
    print(f"Using local model from S3: {model_path}")
else:
    print("[notebook] Model not found in S3, downloading from HuggingFace...")
    from huggingface_hub import snapshot_download

    token = os.getenv("HUGGINGFACE_HUB_TOKEN")
    model_path = snapshot_download(
        repo_id="Qwen/Qwen2.5-1.5B-Instruct",
        local_dir=MODEL_DIR,
        token=token,
        resume_download=True,
        local_dir_use_symlinks=False,
    )
    print(f"Model downloaded to: {model_path}")

In [ ]:
training_runtime_name = os.getenv("TRAINING_RUNTIME")
if not training_runtime_name:
    raise RuntimeError("TRAINING_RUNTIME environment variable is required")

th_runtime = None
for runtime in client.list_runtimes():
    if runtime.name == training_runtime_name:
        th_runtime = runtime
        print("Found runtime: " + str(th_runtime))
        break

if th_runtime is None:
    raise RuntimeError(f"Required runtime '{training_runtime_name}' not found")

In [ ]:
# Submit a training job with valid model + data but an extreme max_batch_len
# value that will cause an OOM crash during the actual torch forward pass.
# The model loads, data loads, torchrun starts the training loop, and then
# the first batch allocation exceeds GPU memory.

from kubeflow.trainer.options.kubernetes import (
    PodTemplateOverrides,
    PodTemplateOverride,
    PodSpecOverride,
    ContainerOverride,
)

LOCAL_MODEL_PATH = "/opt/app-root/src/Qwen/Qwen2.5-1.5B-Instruct"

training_parameters = {
    "model_path": LOCAL_MODEL_PATH,
    "data_path": "/opt/app-root/src/table-gpt-data/train/train_All_100.jsonl",
    "ckpt_output_dir": "/opt/app-root/src/checkpoints",
    "data_output_dir": "/opt/app-root/src/sft-data",
    "effective_batch_size": 128,
    "learning_rate": 5e-6,
    "num_epochs": 1,
    "max_seq_len": 8192,
    # Extreme value: try to pack 10M tokens into a single GPU batch.
    # This will OOM during the forward pass when torch allocates the
    # attention matrices and activations for this massive batch.
    "max_batch_len": 10000000,
}

cache_root = "/opt/app-root/src/.cache/huggingface"
triton_cache = "/opt/app-root/src/.triton"

job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.SFT,
        func_args=training_parameters,
        env={
            "HF_HOME": cache_root,
            "TRITON_CACHE_DIR": triton_cache,
            "XDG_CACHE_HOME": "/opt/app-root/src/.cache",
        },
        resources_per_node={
            "cpu": 4,
            "memory": "32Gi",
            "nvidia.com/gpu": 1,
        },
    ),
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    volumes=[
                        {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {"name": "work", "mountPath": "/opt/app-root/src", "readOnly": False},
                            ],
                        )
                    ],
                ),
            )
        )
    ],
    runtime=th_runtime,
)

print(f"Training job created: {job_name}")

In [ ]:
# Wait for the job to start running, then poll for crash evidence.
# The training pod will start, load the model, begin the training loop,
# and OOM during the first forward pass.
# We check for OOM error strings in logs via get_job_logs() and confirm
# the crash via get_job()/get_job_events() — either the TrainJob reaches
# Failed status or crash-related events (BackOff, OOMKilled) appear.

import time

NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE")

# Event reasons that indicate a container crash or failure
FAILURE_EVENT_REASONS = {"BackOff", "CrashLoopBackOff", "Failed", "OOMKilled", "OOMKilling", "Killing"}

try:
    client.wait_for_job_status(name=job_name, status={"Running", "Failed"}, timeout=300)
except Exception as e:
    print(f"Wait for Running/Failed raised: {e}")

OOM_ERROR_PATTERNS = [
    "OutOfMemoryError",
    "CUDA out of memory",
    "torch.OutOfMemoryError",
    "torch.cuda.OutOfMemoryError",
]

error_found = False
failure_confirmed = False
matched_pattern = None
deadline = time.time() + 600  # 10 minute timeout

while time.time() < deadline:
    # Check logs for OOM error
    if not error_found:
        try:
            log_lines = list(client.get_job_logs(job_name, follow=False))
            log_text = "\n".join(str(line) for line in log_lines)
            for pattern in OOM_ERROR_PATTERNS:
                if pattern in log_text:
                    error_found = True
                    matched_pattern = pattern
                    print(f"Found expected error in logs: '{pattern}'")
                    break
        except Exception as e:
            print(f"Log fetch error (retrying): {e}")

    # Check job failure via get_job() status
    if not failure_confirmed:
        try:
            job = client.get_job(name=job_name)
            if job.status == "Failed":
                failure_confirmed = True
                print(f"Job failure confirmed via get_job(): status={job.status}")
        except Exception as e:
            print(f"get_job() error (retrying): {e}")

    # Check events via get_job_events() for crash signals
    if not failure_confirmed:
        try:
            events = client.get_job_events(name=job_name)
            for event in events:
                reason = getattr(event, "reason", "") or ""
                if reason in FAILURE_EVENT_REASONS:
                    failure_confirmed = True
                    message = getattr(event, "message", "") or ""
                    print(f"Job failure confirmed via get_job_events(): reason={reason} message={message[:100]}")
                    break
        except Exception as e:
            print(f"get_job_events() error (retrying): {e}")

    if error_found and failure_confirmed:
        break

    time.sleep(15)

assert error_found, f"No OOM/crash error found in logs (searched for: {OOM_ERROR_PATTERNS})"
assert failure_confirmed, (
    f"Job failure not confirmed via get_job()/get_job_events() "
    f"— final job status: {client.get_job(name=job_name).status}"
)

print(f"Torchrun failure test PASSED: found '{matched_pattern}' in logs and job failure confirmed via SDK")

In [ ]:
try:
    client.delete_job(job_name)
    print(f"Deleted job: {job_name}")
except Exception as e:
    print(f"Failed to delete job: {e}")

print("NOTEBOOK_STATUS: SUCCESS")